In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("joins").getOrCreate()

In [5]:
customers = [

(1, "Rahul", "Hyderabad"),
(2, "Priya", "Bangalore"),
(3, "Amit", "Mumbai"),
(4, "Sneha", "Chennai"),
(5, "Farhan", "Delhi")

]

customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

customers_df.show()


orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)

]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]

orders_df = spark.createDataFrame(
    orders,
    order_columns
)

orders_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+

+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|     101|          1| Laptop| 65000|
|     102|          2| Mobile| 25000|
|     103|          1|     TV| 45000|
|     104|          3|  Chair|  5000|
|     105|          7|  Watch|  8000|
+--------+-----------+-------+------+



In [6]:
customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

customers_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+



In [7]:
orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)

]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]

In [8]:
orders_df = spark.createDataFrame(
    orders,
    order_columns
)

orders_df.show()

+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|     101|          1| Laptop| 65000|
|     102|          2| Mobile| 25000|
|     103|          1|     TV| 45000|
|     104|          3|  Chair|  5000|
|     105|          7|  Watch|  8000|
+--------+-----------+-------+------+



In [9]:
customers_df.join(orders_df, "customer_id", "inner").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [10]:
customers_df.join(orders_df, "customer_id", "left").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
+-----------+-------------+---------+--------+-------+------+



In [11]:
customers_df.join(orders_df, "customer_id", "right").show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          7|         NULL|     NULL|     105|  Watch|  8000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [12]:
from pyspark.sql.functions import sum

customers_df.join(orders_df, "customer_id").groupBy("customer_name").agg(sum("amount").alias("total_spent")).show()

+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Priya|      25000|
|        Rahul|     110000|
|         Amit|       5000|
+-------------+-----------+



In [13]:
employees = [

(101,"Rahul","IT",75000),
(102,"Priya","IT",85000),
(103,"Amit","IT",65000),

(104,"Sneha","HR",70000),
(105,"Farhan","HR",90000),

(106,"Neha","Finance",95000),
(107,"Arjun","Finance",80000),
(108,"Meera","Finance",75000)

]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary"
]

df = spark.createDataFrame(
    employees,
    columns
)

df.show()

+-----------+-------------+----------+------+
|employee_id|employee_name|department|salary|
+-----------+-------------+----------+------+
|        101|        Rahul|        IT| 75000|
|        102|        Priya|        IT| 85000|
|        103|         Amit|        IT| 65000|
|        104|        Sneha|        HR| 70000|
|        105|       Farhan|        HR| 90000|
|        106|         Neha|   Finance| 95000|
|        107|        Arjun|   Finance| 80000|
|        108|        Meera|   Finance| 75000|
+-----------+-------------+----------+------+



In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [15]:
window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn("row_number", row_number().over(window_spec)).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|row_number|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         6|
|        104|        Sneha|        HR| 70000|         7|
|        103|         Amit|        IT| 65000|         8|
+-----------+-------------+----------+------+----------+



In [16]:
window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn("rank", rank().over(window_spec)).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   7|
|        103|         Amit|        IT| 65000|   8|
+-----------+-------------+----------+------+----+



In [17]:
window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn("rank", dense_rank().over(window_spec)).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   6|
|        103|         Amit|        IT| 65000|   7|
+-----------+-------------+----------+------+----+



In [18]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

df.withColumn("department_rank",rank().over(window_spec)).show()

+-----------+-------------+----------+------+---------------+
|employee_id|employee_name|department|salary|department_rank|
+-----------+-------------+----------+------+---------------+
|        106|         Neha|   Finance| 95000|              1|
|        107|        Arjun|   Finance| 80000|              2|
|        108|        Meera|   Finance| 75000|              3|
|        105|       Farhan|        HR| 90000|              1|
|        104|        Sneha|        HR| 70000|              2|
|        102|        Priya|        IT| 85000|              1|
|        101|        Rahul|        IT| 75000|              2|
|        103|         Amit|        IT| 65000|              3|
+-----------+-------------+----------+------+---------------+

